# Phase 5: Baseline

GitHub issue: #6. Depends on #5 (Phase 4: candidate pool + `score` feature). Evaluated on val only
(test remains locked - see Phase 2 notebook).

## Step 1: Existing baseline

None - stated in Phase 0 (issue #1). No live rule or prior model exists for this offline, synthetic
project.

## Step 2: Simplest reasonable baseline

Rank each session's Phase 4 candidate pool by its co-visitation `score` (no learning, no reranking) -
the pool's own top-50 order *is* the baseline's ranking. This is exactly the comparison point
`_doc/decisions/D2-phase6-scope.md` already committed to: co-visitation baseline vs. one learned
reranker (Phase 6).

In [1]:
from pathlib import Path
import sys

import polars as pl

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
FROZEN_DIR = ROOT / "data" / "frozen"

from ranking.eval.bootstrap import bootstrap_ci
from ranking.eval.metrics import ndcg_at_k

split = "val"
examples = pl.read_parquet(FROZEN_DIR / "ranking_examples.parquet").filter(pl.col("split") == split)
features = pl.read_parquet(FROZEN_DIR / "features" / f"{split}.parquet")

per_session_gains = (
    features.sort(["session", "rank"])
    .group_by("session", maintain_order=True)
    .agg(pl.col("grade").alias("gains"))
)
joined = examples.select("session").join(per_session_gains, on="session", how="left")
n_missing = joined["gains"].null_count()
print(f"{examples.height:,} examples, {n_missing:,} with an empty candidate pool")

396,786 examples, 21,586 with an empty candidate pool


Sessions with an empty candidate pool (no co-visitation neighbours survived pruning/frequency
filtering) count as NDCG=0 - a real, honest result, not excluded from the denominator.

In [2]:
ndcg_scores = [
    ndcg_at_k(gains, k=10) if gains is not None else 0.0
    for gains in joined["gains"].to_list()
]
mean, lower, upper = bootstrap_ci(ndcg_scores, n_resamples=10_000, seed=0)
print(f"NDCG@10: {mean:.4f}  95% CI [{lower:.4f}, {upper:.4f}]  (n={len(ndcg_scores):,})")

NDCG@10: 0.0335  95% CI [0.0330, 0.0339]  (n=396,786)


## Result

**NDCG@10 = 0.0335, 95% bootstrap CI [0.0330, 0.0339]** on val.

This is low in absolute terms, and that's expected, not a bug: Phase 4 found only ~11% of val examples
even have the true item anywhere in the top-50 pool, and most of the label distribution is click
(grade=1, the lowest graded value) even when the item is found - both drag the mean NDCG@10 down hard.
This is the honest floor Phase 6's reranker has to beat, not a measurement error.

## Options considered

- Majority-class / most-popular-item baseline: not run - co-visitation is strictly more informed (it
  already conditions on the session's own prefix) and matches what the archived plan already validated
  as the right non-learned baseline.
- Logistic regression on a few features: rejected for the baseline role - co-visitation-only ranking is
  the more standard "non-learned floor" for a candidate-generation task, and Phase 6 already covers the
  "one learned model" comparison point.

## Exit criteria

Baseline NDCG@10 on val, recorded with the Phase 1 snapshot version and Phase 2 split version. Met.